<a href="https://colab.research.google.com/github/AishwaryaChennadi/DataScience/blob/main/API_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import requests
import os
import json
from datetime import datetime

LOG_FILE = "test_results.log"

# --- Step 1: API Client ---

def get_data(endpoint="/posts"):

    url = f"https://jsonplaceholder.typicode.com{endpoint}"
    try:
        response = requests.get(url, timeout=5)
        return response   # return even if status != 200
    except requests.exceptions.RequestException:
        return None
# --- Step 2: Validators ---
def validate_status(response):
    """Check if status code = 200"""
    if response and response.status_code == 200:
        return True, "Status code is 200"
    return False, f"Unexpected status code: {response.status_code if response else 'No Response'}"

def validate_schema(response):
    """Validate response structure and data types"""
    try:
        data = response.json()
        if isinstance(data, list) and len(data) > 0:
            sample = data[0]
            required_keys = ["userId", "id", "title", "body"]
            for key in required_keys:
                if key not in sample:
                    return False, f"Missing key: {key}"
            if not isinstance(sample["id"], int):
                return False, "id is not int"
            if not isinstance(sample["title"], str):
                return False, "title is not str"
            return True, "Schema validated successfully"
        return False, "Response is not a valid list"
    except Exception as e:
        return False, f"Schema validation failed: {str(e)}"

# --- Step 3: Logger ---
def log_result(test_name, status, message):
    """Log test results to file"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result_line = f"[{timestamp}] {test_name} - {'PASS' if status else 'FAIL'} - {message}\n"
    with open(LOG_FILE, "a") as f:
        f.write(result_line)
    print(result_line.strip())

# --- Step 4: Runner ---
def run_tests():
    print("Running API Tests...\n")

    # Test 1: Valid endpoint
    response = get_data("/posts")
    status, msg = validate_status(response)
    log_result("Test 1: Status Code", status, msg)

    status, msg = validate_schema(response)
    log_result("Test 2: Schema Validation", status, msg)

    # Test 2: Invalid endpoint
    bad_response = get_data("/invalid")
    status, msg = validate_status(bad_response)
    log_result("Test 3: Invalid Endpoint", status, msg)

if __name__ == "__main__":
    # Clear old logs
    if os.path.exists(LOG_FILE):
        os.remove(LOG_FILE)
    run_tests()


Running API Tests...

[2026-05-11 06:25:21] Test 1: Status Code - PASS - Status code is 200
[2026-05-11 06:25:21] Test 2: Schema Validation - PASS - Schema validated successfully
[2026-05-11 06:25:21] Test 3: Invalid Endpoint - FAIL - Unexpected status code: No Response


In [2]:
!mkdir -p /content/week2_project

In [3]:
!touch /content/week2_project/api_client.py
!touch /content/week2_project/validators.py
!touch /content/week2_project/logger.py
!touch /content/week2_project/README.md

In [4]:
!mkdir -p /content/week2_project/tests

In [6]:
with open("/content/week2_project/api_client.py", "w") as f:
    f.write("""import requests

def get_data(endpoint="/posts"):
    \"\"\"Fetch data from JSONPlaceholder API\"\"\"
    url = f"https://jsonplaceholder.typicode.com{endpoint}"
    try:
        response = requests.get(url, timeout=5)
        return response
    except requests.exceptions.RequestException:
        return None
""")


In [7]:
with open("/content/week2_project/validators.py", "w") as f:
    f.write("""def validate_status(response):
    if response and response.status_code == 200:
        return True, "Status code is 200"
    return False, f"Unexpected status code: {response.status_code if response else 'No Response'}"

def validate_schema(response):
    try:
        data = response.json()
        if isinstance(data, list) and len(data) > 0:
            sample = data[0]
            required_keys = ["userId", "id", "title", "body"]
            for key in required_keys:
                if key not in sample:
                    return False, f"Missing key: {key}"
            if not isinstance(sample["id"], int):
                return False, "id is not int"
            if not isinstance(sample["title"], str):
                return False, "title is not str"
            return True, "Schema validated successfully"
        return False, "Response is not a valid list"
    except Exception as e:
        return False, f"Schema validation failed: {str(e)}"
""")


In [8]:
with open("/content/week2_project/logger.py", "w") as f:
    f.write("""import logging

logging.basicConfig(
    filename="test_results.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

def log_result(test_name, status, message):
    if status:
        logging.info(f"{test_name} - PASS - {message}")
    else:
        logging.error(f"{test_name} - FAIL - {message}")
""")


In [9]:
with open("/content/week2_project/tests/test_api.py", "w") as f:
    f.write("""import pytest
from api_client import get_data
from validators import validate_status, validate_schema
from logger import log_result

@pytest.fixture
def valid_response():
    return get_data("/posts")

@pytest.fixture
def invalid_response():
    return get_data("/invalid")

def test_status_code(valid_response):
    status, msg = validate_status(valid_response)
    log_result("Test Status Code", status, msg)
    assert status, msg

def test_schema(valid_response):
    status, msg = validate_schema(valid_response)
    log_result("Test Schema Validation", status, msg)
    assert status, msg

def test_invalid_endpoint(invalid_response):
    status, msg = validate_status(invalid_response)
    log_result("Test Invalid Endpoint", status, msg)
    assert not status, "Expected failure for invalid endpoint"
""")


In [15]:
!pip install pytest


In [11]:
!ls /content/week2_project



api_client.py  logger.py  README.md  tests  validators.py


In [12]:
!touch /content/week2_project/__init__.py
!touch /content/week2_project/tests/__init__.py


In [14]:
%cd /content
!pytest -v week2_project/tests/test_api.py

/content
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.34
collected 3 items                                                              

week2_project/tests/test_api.py::test_status_code PASSED                 [ 33%]
week2_project/tests/test_api.py::test_schema PASSED                      [ 66%]
week2_project/tests/test_api.py::test_invalid_endpoint PASSED            [100%]

============================== 3 passed in 0.13s ===============================


In [17]:
with open("/content/week2_project/api_client.py", "w") as f:
    f.write("""import requests

def get_data(endpoint="/posts"):
    \"\"\"Fetch data from JSONPlaceholder API\"\"\"
    url = f"https://jsonplaceholder.typicode.com{endpoint}"
    try:
        response = requests.get(url, timeout=5)
        return response   # return even if status != 200
    except requests.exceptions.RequestException:
        return None
""")


In [18]:
from week2_project.api_client import get_data
resp = get_data("/invalid")
print(resp.status_code)   # should print 404


404


In [19]:
%cd /content
!pytest -v week2_project/tests/test_api.py


/content
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.34
collected 3 items                                                              

week2_project/tests/test_api.py::test_status_code PASSED                 [ 33%]
week2_project/tests/test_api.py::test_schema PASSED                      [ 66%]
week2_project/tests/test_api.py::test_invalid_endpoint PASSED            [100%]

============================== 3 passed in 0.11s ===============================
